**Purpose:** FINAL news sentiment model, TF-IDF embeddings variant (weights gitignored — retrain here).

**Inputs:** `/kaggle/input/news-sentiment/dataset.parquet`, `dataset.parquet`

**Outputs:** `vf2_tfidf_embeddings/final_model.pt`, `vf2_tfidf_embeddings/test_confusion_matrix.png`, `vf2_tfidf_embeddings/train_confusion_matrix.png`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT
from src.seeds import SEED_SENTIMENT_NEWS_VF2_TFIDF_EMBEDDINGS


In [ ]:
# 🔹 FinBERT backbone
# Total: 109,482,240
# Trainable: 14,175,744

# 🔹 TF-IDF projection head
# Total: 19,864
# Trainable: 19,864

# 🔹 Classifier
# Total: 2,499
# Trainable: 2,499

# ======================================================================
# TOTAL PARAMETERS:     109,504,603 (100%)
# TRAINABLE PARAMETERS: 14,198,107 (13.0%)
# FROZEN PARAMETERS:    95,306,496 (87.0%)
# ======================================================================

In [ ]:
!pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 --index-url https://download.pytorch.org/whl/cu118

!pip install transformers==4.36.2 scikit-learn pandas numpy

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected")

# =========================================================
# Sector Descriptions
# =========================================================

# https://www.msci.com/downloads/documents/indexes/gics/GICS+Sector+Definitions+2023.pdf

sector_desc = {
    "Energy": (
        "The Energy Sector comprises companies engaged in exploration & "
        "production, refining & marketing, and storage & transportation of oil "
        "and gas and coal & consumable fuels. It also includes companies that "
        "offer oil & gas equipment and services."
    ),
    "Materials": (
        "The Materials Sector includes companies that manufacture chemicals, "
        "construction materials, forest products, glass, paper and related "
        "packaging products, and metals, minerals and mining companies, "
        "including producers of steel."
    ),
    "Industrials": (
        "The Industrials Sector includes manufacturers and distributors of "
        "capital goods such as aerospace & defense, building products, "
        "electrical equipment and machinery and companies that offer "
        "construction & engineering services. It also includes providers of "
        "commercial & professional services and transportation services."
    ),
    "Consumer Discretionary": (
        "The Consumer Discretionary Sector encompasses those businesses that "
        "tend to be the most sensitive to economic cycles. Its manufacturing "
        "segment includes automobiles & components, household durable goods, "
        "leisure products, and textiles & apparel. The services segment "
        "includes hotels, restaurants, and other leisure facilities, as well as "
        "distributors and retailers of consumer discretionary products."
    ),
    "Consumer Staples": (
        "The Consumer Staples Sector comprises companies whose businesses are "
        "less sensitive to economic cycles. It includes manufacturers and "
        "distributors of food, beverages and tobacco and producers of non-durable "
        "household goods and personal products. It also includes distributors "
        "and retailers of consumer staples products including food & drug "
        "retailing companies."
    ),
    "Health Care": (
        "The Health Care Sector includes health care providers and services, "
        "companies that manufacture and distribute health care equipment and "
        "supplies, health care technology firms, and companies involved in "
        "pharmaceuticals and biotechnology research, development, production, "
        "and marketing."
    ),
    "Financials": (
        "The Financials Sector contains companies engaged in banking, financial "
        "services, consumer finance, capital markets, and insurance activities. "
        "It also includes financial exchanges, data services, and mortgage real "
        "estate investment trusts."
    ),
    "Information Technology": (
        "The Information Technology Sector comprises companies offering "
        "software and technology services, and manufacturers and distributors "
        "of technology hardware and equipment including communications "
        "equipment, computers, peripherals, and semiconductor products."
    ),
    "Communication Services": (
        "The Communication Services Sector includes companies that facilitate "
        "communication and offer related content and information through various "
        "mediums, including telecommunications, media and entertainment, and "
        "interactive services and platforms."
    ),
    "Utilities": (
        "The Utilities Sector comprises utility companies such as electric, "
        "gas, and water utilities, as well as independent power producers, "
        "energy traders, and companies involved in renewable generation."
    ),
    "Real Estate": (
        "The Real Estate Sector contains companies engaged in real estate "
        "development and operations as well as firms offering real estate-"
        "related services, including equity real estate investment trusts."
    ),
}

In [ ]:
import pandas as pd
import json
import numpy as np

#df_raw = pd.read_parquet("/kaggle/input/news-sentiment/dataset.parquet")
df_raw = pd.read_parquet("dataset.parquet")

#with open("/kaggle/input/news-sentiment/sector_sentiment_quotations.json", "r") as f:
with open(str(PROJECT_ROOT / "01_data/processed/sector_sentiment_quotations.json"), "r") as f:
    sector_sentiment_quotations = json.load(f)

gpt_testset_ids = []
for sector in sector_sentiment_quotations:
    for expected_sentiment in sector_sentiment_quotations[sector]:
        for _ in sector_sentiment_quotations[sector][expected_sentiment]:
            for quote in sector_sentiment_quotations[sector][expected_sentiment][_]:
                if quote["label"] == expected_sentiment:
                    gpt_testset_ids.append(quote["gkrecordid"] + " (" + sector + ")")

train_df = df_raw[~df_raw["GKGRECORDID"].isin(gpt_testset_ids)].copy()
min_count = train_df.value_counts(["sector", "majority_sentiment"]).values.min()
df = (
    train_df
    .groupby(["sector", "majority_sentiment"])
    .apply(lambda x: x.sample(min_count, random_state=SEED_SENTIMENT_NEWS_VF2_TFIDF_EMBEDDINGS))
    .reset_index(drop=True)
)

test_df = df_raw[df_raw["GKGRECORDID"].isin(gpt_testset_ids)].copy()

from sklearn.feature_extraction.text import TfidfVectorizer

sector_desc_df = pd.DataFrame({
    "sector": sector_desc.keys(),
    "sector_description": sector_desc.values()
})


tfidf = TfidfVectorizer(max_features=300, ngram_range=(1,2), stop_words="english")
sector_matrix = tfidf.fit_transform(sector_desc_df["sector_description"])

tfidf_cols = [f"tfidf_{i}" for i in range(sector_matrix.shape[1])]

sector_tfidf = pd.DataFrame(
    sector_matrix.toarray(),
    index=sector_desc_df["sector"],
    columns=tfidf_cols
).astype(np.float32)


# df[tfidf_cols] = df[tfidf_cols].astype(np.float32)
# test_df = test_df.merge(sector_tfidf, left_on="sector", right_index=True)
# test_df[tfidf_cols] = test_df[tfidf_cols].astype(np.float32)

df = df.merge(sector_tfidf, left_on="sector", right_index=True)
test_df = test_df.merge(sector_tfidf, left_on="sector", right_index=True)




# =========================================================
# FINBERT + FEATURE SENTIMENT CLASSIFIER (FULL PIPELINE)
# =========================================================

import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, AdamW
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ================= CONFIG =================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ProsusAI/finbert"
TEXT_FEATURE = "sector" 
MAX_LEN = 128


# ================= DATA (df must already exist) =================
label2id = {l:i for i,l in enumerate(sorted(df["majority_sentiment"].unique()))}
df["label"] = df["majority_sentiment"].map(label2id)

# ================= DATASET =================
class SentDataset(Dataset):
    def __init__(self, df, tokenizer, tfidf_cols):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.tfidf_cols = tfidf_cols

    def __len__(self):
        return len(self.df)   # ← MISSING

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["quotation"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "tfidf": torch.from_numpy(row[self.tfidf_cols].to_numpy(dtype=np.float32)),
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
        }


# ================= MODEL =================
class FinBERTWithTFIDF(nn.Module):
    def __init__(self, num_labels, tfidf_dim, proj_dim, dropout, unfrozen):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)


        for name, param in self.bert.named_parameters():
            if "encoder.layer." in name:
                layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
        
                if layer_num < unfrozen:        # freeze lower layers
                    param.requires_grad = False
                else:                    # unfreeze 9, 10, 11...
                    param.requires_grad = True
            else:
                param.requires_grad = False  # embeddings, pooler etc. (optional)

        self.tfidf_proj = nn.Sequential(
            nn.LayerNorm(tfidf_dim),
            nn.Linear(tfidf_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size + proj_dim, num_labels)

    def forward(self, input_ids, attention_mask, tfidf):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0]
        tfidf_feat = self.tfidf_proj(tfidf)
        x = torch.cat([cls, tfidf_feat], dim=1)
        x = self.dropout(x)
        return self.classifier(x)


# ================= HYPERPARAMETERS =================

hp = {"unfrozen_gte":10,"bert_lr":5e-6,"head_lr":5e-5,"dropout":0.2,"proj_dim":64,"tfidf_dim":300,"weight_decay": 0.01,"bs":32,"epochs":8}

# ================= FINAL TRAINING (FULL TRAIN SET) =================

os.makedirs("vf2_tfidf_embeddings", exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SentDataset(df, tokenizer, tfidf_cols),
                          batch_size=hp["bs"], shuffle=True)

test_df["label"] = test_df["majority_sentiment"].map(label2id)
test_loader = DataLoader(SentDataset(test_df, tokenizer, tfidf_cols),
                         batch_size=hp["bs"])

model = FinBERTWithTFIDF(
    len(label2id),
    len(tfidf_cols),
    hp["proj_dim"],
    hp["dropout"],
    hp["unfrozen_gte"]
).to(DEVICE)

opt = AdamW([
    {"params": filter(lambda p: p.requires_grad, model.bert.parameters()), "lr": hp["bert_lr"], "weight_decay": hp["weight_decay"]},
    {"params": model.tfidf_proj.parameters(), "lr": hp["head_lr"], "weight_decay": hp["weight_decay"]},
    {"params": model.classifier.parameters(), "lr": hp["head_lr"], "weight_decay": hp["weight_decay"]},
])


loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

train_losses = []
test_losses = []

print("\n🚀 Training Final Model on Full Training Set\n")

for epoch in range(1, hp["epochs"]+1):
    model.train()
    epoch_train_losses = []

    for batch in tqdm(train_loader):
        for k in batch:
            batch[k] = batch[k].to(DEVICE)

        opt.zero_grad()
        logits = model(batch["input_ids"], batch["attention_mask"], batch["tfidf"])
        loss = loss_fn(logits, batch["label"])
        loss.backward()
        opt.step()
        epoch_train_losses.append(loss.item())

    train_loss = np.mean(epoch_train_losses)
    train_losses.append(train_loss)

    # ===== TEST LOSS EACH EPOCH =====
    model.eval()
    epoch_test_losses = []

    with torch.no_grad():
        for batch in test_loader:
            for k in batch:
                batch[k] = batch[k].to(DEVICE)
            logits = model(batch["input_ids"], batch["attention_mask"], batch["tfidf"])
            loss = loss_fn(logits, batch["label"])
            epoch_test_losses.append(loss.item())

    test_loss = np.mean(epoch_test_losses)
    test_losses.append(test_loss)

    print(f"Epoch {epoch}/{hp['epochs']} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")


# ================= TRAIN EVALUATION =================

model.eval()
train_preds, train_labels = [], []

with torch.no_grad():
    for batch in train_loader:
        for k in batch:
            batch[k] = batch[k].to(DEVICE)

        logits = model(batch["input_ids"], batch["attention_mask"], batch["tfidf"])
        preds = torch.argmax(logits, dim=1)

        train_preds.extend(preds.cpu().tolist())
        train_labels.extend(batch["label"].cpu().tolist())

print("\n📊 Train Classification Report:\n")
train_report = classification_report(train_labels, train_preds, target_names=label2id.keys())
print(train_report)

with open("vf2_tfidf_embeddings/train_report.txt", "w") as f:
    f.write(train_report)


# ================= TEST EVALUATION =================

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        for k in batch:
            batch[k] = batch[k].to(DEVICE)

        logits = model(batch["input_ids"], batch["attention_mask"], batch["tfidf"])
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["label"].cpu().tolist())

print("\n📊 Test Classification Report:\n")
report_str = classification_report(all_labels, all_preds, target_names=label2id.keys())
print(report_str)

with open("vf2_tfidf_embeddings/test_report.txt", "w") as f:
    f.write(report_str)

# ================= CONFUSION MATRIX =================

train_cm = confusion_matrix(train_labels, train_preds)
train_cm_percent = train_cm.astype("float") / train_cm.sum(axis=1)[:, np.newaxis] * 100

fig, ax = plt.subplots(figsize=(7,6))
disp = ConfusionMatrixDisplay(train_cm, display_labels=label2id.keys())
disp.plot(values_format="d", cmap="Greens", ax=ax)

for i in range(train_cm.shape[0]):
    for j in range(train_cm.shape[1]):
        ax.text(j, i, f"\n{train_cm_percent[i,j]:.1f}%", ha="center", va="center", color="red", fontsize=9)

plt.title("Train Confusion Matrix (Counts + %)")
plt.savefig("vf2_tfidf_embeddings/train_confusion_matrix.png", bbox_inches="tight")
plt.show()



cm = confusion_matrix(all_labels, all_preds)
cm_percent = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100

fig, ax = plt.subplots(figsize=(7,6))
disp = ConfusionMatrixDisplay(cm, display_labels=label2id.keys())
disp.plot(values_format="d", cmap="Blues", ax=ax)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"\n{cm_percent[i,j]:.1f}%", ha="center", va="center", color="red", fontsize=9)

plt.title("Test Confusion Matrix (Counts + %)")
plt.savefig("vf2_tfidf_embeddings/test_confusion_matrix.png", bbox_inches="tight")
plt.show()

# ================= SAVE MODEL =================

torch.save(model.state_dict(), "vf2_tfidf_embeddings/final_model.pt")

print("\n✅ Final model + evaluation saved in /vf2 tfidfemb/")

# ================= FURTHER RESULTS =================
def cm_to_text(cm, labels):
    cm_percent = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100
    lines = []

    header = "Pred →     " + "  ".join([f"{l:>8}" for l in labels])
    lines.append(header)
    lines.append("-" * len(header))

    for i, l in enumerate(labels):
        row_counts = "  ".join([f"{cm[i,j]:>8}" for j in range(len(labels))])
        row_pct = "  ".join([f"{cm_percent[i,j]:>7.1f}%" for j in range(len(labels))])
        lines.append(f"True {l:<8} | {row_counts}")
        lines.append(f"           | {row_pct}")
        lines.append("")

    return "\n".join(lines)

def collect_preds(loader, df_source):
    model.eval()
    preds, labels, sectors = [], [], []

    idx_offset = 0  # keeps track of row position

    with torch.no_grad():
        for batch in loader:
            bs = batch["label"].size(0)

            for k in ["input_ids", "attention_mask", "tfidf", "label"]:
                batch[k] = batch[k].to(DEVICE)

            logits = model(batch["input_ids"], batch["attention_mask"], batch["tfidf"])
            p = torch.argmax(logits, dim=1)

            preds.extend(p.cpu().tolist())
            labels.extend(batch["label"].cpu().tolist())

            # grab sectors directly from original dataframe slice
            sectors.extend(df_source.iloc[idx_offset:idx_offset+bs]["sector"].tolist())
            idx_offset += bs

    return preds, labels, sectors



train_preds, train_labels, train_sectors = collect_preds(train_loader, df)
test_preds, test_labels, test_sectors = collect_preds(test_loader, test_df)

id2label = {v:k for k,v in label2id.items()}

unique_sectors = sorted(set(train_sectors + test_sectors))

print("\n📊 Generating PER-SECTOR evaluation files...\n")

for sec_id in unique_sectors:
    safe_name = sec_id.replace(" ", "_").replace("/", "_")

    # ---- TRAIN FILTER ----
    tr_idx = [i for i,s in enumerate(train_sectors) if s == sec_id]
    te_idx = [i for i,s in enumerate(test_sectors) if s == sec_id]

    if len(tr_idx) == 0 or len(te_idx) == 0:
        continue

    tr_labels = [train_labels[i] for i in tr_idx]
    tr_preds  = [train_preds[i] for i in tr_idx]

    te_labels = [test_labels[i] for i in te_idx]
    te_preds  = [test_preds[i] for i in te_idx]

    # ---- REPORTS ----
    train_report = classification_report(tr_labels, tr_preds, target_names=id2label.values())
    test_report  = classification_report(te_labels, te_preds, target_names=id2label.values())

    # ---- CONFUSION MATRICES ----
    train_cm = confusion_matrix(tr_labels, tr_preds)
    test_cm  = confusion_matrix(te_labels, te_preds)

    train_cm_txt = cm_to_text(train_cm, id2label.values())
    test_cm_txt  = cm_to_text(test_cm, id2label.values())

    # ---- WRITE FILE ----
    with open(f"vf2 tfidfemb/sector_{safe_name}_report.txt", "w") as f:
        f.write(f"SECTOR: {safe_name}\n")
        f.write("="*60 + "\n\n")

        f.write("TRAIN CLASSIFICATION REPORT\n")
        f.write("-"*60 + "\n")
        f.write(train_report + "\n\n")

        f.write("TRAIN CONFUSION MATRIX (Counts + %)\n")
        f.write("-"*60 + "\n")
        f.write(train_cm_txt + "\n\n")

        f.write("TEST CLASSIFICATION REPORT\n")
        f.write("-"*60 + "\n")
        f.write(test_report + "\n\n")

        f.write("TEST CONFUSION MATRIX (Counts + %)\n")
        f.write("-"*60 + "\n")
        f.write(test_cm_txt)

print("✅ Per-sector reports saved in /vf2 tfidfemb/")
